Install the pandas package.

In [1]:
%pip install pandas


Note: you may need to restart the kernel to use updated packages.


This loads the summary statistics file from a study into a pandas dataframe and inspects the dimensions and first few rows.

In [ ]:
import pandas as pd
# Update this path to whichever CSV you want to validate.
INPUT_CSV = "/Users/aanikaschueler/Desktop/Beiwe/beiwe/code/forest_mano/Data Volumes and Summary Statistics - BU_Fulford_ Smartphone sensing of social activity clinical - 2026-06-26 21_54 (UTC).csv"
df = pd.read_csv(INPUT_CSV)
print(df.shape)
print(df.head())

(50507, 68)
         Date Participant Id                  Study Id Timezone  \
0  2022-07-06       163lrwb6  UZ3mQzoQGo2TPEzWYw2z5dTT      EDT   
1  2022-07-07       163lrwb6  UZ3mQzoQGo2TPEzWYw2z5dTT      EDT   
2  2022-07-08       163lrwb6  UZ3mQzoQGo2TPEzWYw2z5dTT      EDT   
3  2022-07-09       163lrwb6  UZ3mQzoQGo2TPEzWYw2z5dTT      EDT   
4  2022-07-10       163lrwb6  UZ3mQzoQGo2TPEzWYw2z5dTT      EDT   

   Accelerometer Bytes  Ambient Audio Bytes  App Log Bytes  Bluetooth Bytes  \
0                  NaN                  NaN            NaN              NaN   
1                  NaN                  NaN            NaN              NaN   
2                  NaN                  NaN            NaN              NaN   
3                  NaN                  NaN            NaN              NaN   
4                  NaN                  NaN            NaN              NaN   

   Calls Bytes  Devicemotion Bytes  ...  Outgoing Call Degree  \
0          NaN                 NaN  ...      

This defines the mapping between each raw Beiwe data stream and the summary statistics that should be generated from that stream. The mappings are used to validate whether summary metrics are present when corresponding raw data exists.

In [ ]:
# Observation-time fields are excluded because they may be generated independently from the core GPS movement metrics.
STREAMS = {
    "GPS": {
    "raw_col": "Gps Bytes",
    "metric_cols": [
        "Distance Diameter",
        "Distance From Home",
        "Distance Traveled",
        "Flight Distance Average",
        "Flight Distance Stddev",
        "Flight Duration Average",
        "Flight Duration Stddev",
        "Home Duration",
        "Gyration Radius",
        "Significant Location Count",
        "Significant Location Entropy",
        "Total Flight Time",
    ],
},
    "Accelerometer": {
        "raw_col": "Accelerometer Bytes",
        "metric_cols": ["Walking Time", "Steps", "Cadence"],
    },
    "Calls": {
        "raw_col": "Calls Bytes",
        "metric_cols": [
            "Incoming Call Count",
            "Incoming Call Degree",
            "Incoming Call Duration",
            "Outgoing Call Count",
            "Outgoing Call Degree",
            "Outgoing Call Duration",
            "Missed Call Count",
            "Missed Callers",
        ],
    },
    "Texts": {
        "raw_col": "Texts Bytes",
        "metric_cols": [
            "Incoming Text Count",
            "Incoming Text Degree",
            "Incoming Text Length",
            "Outgoing Text Count",
            "Outgoing Text Degree",
            "Outgoing Text Length",
            "Incoming Text Reciprocity",
            "Outgoing Text Reciprocity",
            "Outgoing Mms Count",
            "Incoming Mms Count",
        ],
    },
}

This creates a validation function that compares raw data availability against summary statistic availability for a given data stream. Specifically, this function identifies days with raw data present (raw_col > 0) and checks if all expected summary metrics are available. Then, it returns a report containing the participant ID, date, stream name, and issue type if there are any flags.

In [3]:
def check_stream(df, stream_name, raw_col, metric_cols):
    raw_present = df[raw_col].fillna(0) > 0
# A metric of 0 is considered present. Only a blank or NaN value is considered missing. 
    metrics_present = df[metric_cols].notna().all(axis=1)

    issue = pd.Series([None] * len(df), index=df.index)

    issue[raw_present & ~metrics_present] = "Raw data present but metrics missing"
    issue[~raw_present & metrics_present] = "Metrics present but raw data missing"

    report = df.loc[issue.notna(), ["Participant Id", "Date"]].copy()
    report["Stream"] = stream_name
    report["Issue"] = issue[issue.notna()].values

    return report

This chunk runs the validation function across all the streams and combines the result into a single report . It also summarizes the number of inconsistencies that were detected.

In [4]:
reports = []

for stream_name, config in STREAMS.items():
    stream_report = check_stream(
        df,
        stream_name,
        config["raw_col"],
        config["metric_cols"]
    )
    reports.append(stream_report)

final_report = pd.concat(reports, ignore_index=True)

print(final_report.shape)
print(final_report)

(501, 4)
    Participant Id        Date Stream                                 Issue
0         163lrwb6  2023-05-17    GPS  Raw data present but metrics missing
1         1akm4vyi  2023-05-14    GPS  Raw data present but metrics missing
2         9fn8c4k9  2024-03-23    GPS  Raw data present but metrics missing
3         bir3qzk4  2022-08-10    GPS  Raw data present but metrics missing
4         cqbxn7in  2023-06-01    GPS  Raw data present but metrics missing
..             ...         ...    ...                                   ...
496       zvbn42zg  2023-04-28  Texts  Metrics present but raw data missing
497       zvbn42zg  2023-04-29  Texts  Metrics present but raw data missing
498       zxie5nmv  2023-04-29  Texts  Metrics present but raw data missing
499       zxie5nmv  2023-05-06  Texts  Metrics present but raw data missing
500       zxie5nmv  2023-05-08  Texts  Metrics present but raw data missing

[501 rows x 4 columns]


Ensuring Expected Columns are Present

In [13]:
expected_columns = [
    "Date",
    "Participant Id",
    "Study Id",
    "Timezone",
    "Accelerometer Bytes",
    "Ambient Audio Bytes",
    "App Log Bytes",
    "Bluetooth Bytes",
    "Calls Bytes",
    "Devicemotion Bytes",
    "Gps Bytes",
    "Gyro Bytes",
    "Identifiers Bytes",
    "Ios Log Bytes",
    "Magnetometer Bytes",
    "Power State Bytes",
    "Proximity Bytes",
    "Reachability Bytes",
    "Survey Answers Bytes",
    "Survey Timings Bytes",
    "Texts Bytes",
    "Audio Recordings Bytes",
    "Wifi Bytes",
    "Distance Diameter",
    "Distance From Home",
    "Distance Traveled",
    "Flight Distance Average",
    "Flight Distance Stddev",
    "Flight Duration Average",
    "Flight Duration Stddev",
    "Home Duration",
    "Gyration Radius",
    "Significant Location Count",
    "Significant Location Entropy",
    "Pause Time",
    "Obs Duration",
    "Obs Day",
    "Obs Night",
    "Total Flight Time",
    "Av Pause Duration",
    "Sd Pause Duration",
    "Physical Circadian Rhythm",
    "Physical Circadian Rhythm Stratified",
    "Incoming Text Count",
    "Incoming Text Degree",
    "Incoming Text Length",
    "Outgoing Text Count",
    "Outgoing Text Degree",
    "Outgoing Text Length",
    "Incoming Text Reciprocity",
    "Outgoing Text Reciprocity",
    "Outgoing Mms Count",
    "Incoming Mms Count",
    "Mean Responsiveness Text",
    "Incoming Call Count",
    "Incoming Call Degree",
    "Incoming Call Duration",
    "Outgoing Call Count",
    "Outgoing Call Degree",
    "Outgoing Call Duration",
    "Missed Call Count",
    "Missed Callers",
    "Mean Responsiveness Call",
    "Call Reciprocity",
    "Uniq Individual Call Or Text Count",
    "Walking Time",
    "Steps",
    "Cadence"
]

missing_columns = [col for col in expected_columns if col not in df.columns]
extra_columns = [col for col in df.columns if col not in expected_columns]

print("===== Column Schema Check =====")

if not missing_columns and not extra_columns:
    print("PASS: All expected columns are present.")
else:
    if missing_columns:
        print("\nMissing columns:")
        for col in missing_columns:
            print(f"  - {col}")

    if extra_columns:
        print("\nUnexpected columns:")
        for col in extra_columns:
            print(f"  - {col}")

===== Column Schema Check =====
PASS: All expected columns are present.


Data Type Validation - Numeric

In [6]:
# Build the numeric column list from the  defined stream mappings.

stream_numeric_columns = []

for config in STREAMS.values():
    stream_numeric_columns.append(config["raw_col"])
    stream_numeric_columns.extend(config["metric_cols"])

additional_numeric_columns = [
    "Pause Time",
    "Obs Duration",
    "Obs Day",
    "Obs Night",
    "Av Pause Duration",
    "Sd Pause Duration",
    "Physical Circadian Rhythm",
    "Physical Circadian Rhythm Stratified",
]

numeric_columns = list(
    dict.fromkeys(stream_numeric_columns + additional_numeric_columns)
)

# Only check columns that actually exist in the current export.
numeric_columns = [
    col for col in numeric_columns
    if col in df.columns
]

incorrect_types = []

for col in numeric_columns:
    if not pd.api.types.is_numeric_dtype(df[col]):
        incorrect_types.append((col, str(df[col].dtype)))

print("===== Data Type Validation =====")

if not incorrect_types:
    print("PASS: All expected numeric columns have numeric data types.")
else:
    print("FAIL: The following columns have incorrect data types:")

    for col, dtype in incorrect_types:
        print(f"  - {col}: {dtype}")

===== Data Type Validation =====
PASS: All expected numeric columns have numeric data types.


Range Checks - Non-Negative Values

In [7]:
negative_value_summary = {}

for col in numeric_columns:
    negative_count = (df[col] < 0).sum()
    
    if negative_count > 0:
        negative_value_summary[col] = negative_count

print("===== Range Checks: Non-negative Values =====")

if not negative_value_summary:
    print("PASS: No negative values found in numeric metric columns.")
else:
    print("FAIL: Negative values found:")
    for col, count in negative_value_summary.items():
        print(f"  - {col}: {count} negative values")

===== Range Checks: Non-negative Values =====
PASS: No negative values found in numeric metric columns.


The following chunk may be run if the range check failed and negative values are found. It will show which rows have negative values.

In [17]:
# Show rows with negative values

for col in numeric_columns:
    bad_rows = df[df[col] < 0]
    
    if not bad_rows.empty:
        print(f"\nNegative values in {col}:")
        print(bad_rows[["Participant Id", "Date", col]])

Physical Bounds: This check identifies duration variables that fall outside their natural range. Daily duration variables must be between 0 and 24 hours.

In [8]:
duration_bounds = {
    "Home Duration": (0, 24),
    "Obs Day": (0, 24),
    "Obs Night": (0, 24),
    "Obs Duration": (0, 24),
}

physical_bound_issues = []

print("===== Physical Bounds Check =====")

for col, (lower_bound, upper_bound) in duration_bounds.items():
    if col not in df.columns:
        print(f"SKIP: {col} is not present in the file.")
        continue

    values = pd.to_numeric(df[col], errors="coerce")

    invalid_mask = (
        values.notna()
        & (
            (values < lower_bound)
            | (values > upper_bound)
        )
    )

    invalid_rows = df.loc[
        invalid_mask,
        ["Participant Id", "Date", col]
    ].copy()

    if not invalid_rows.empty:
        invalid_rows["Check"] = "Physical Bounds"
        invalid_rows["Issue"] = (
            f"{col} must be between "
            f"{lower_bound} and {upper_bound} hours"
        )

        physical_bound_issues.append(invalid_rows)

if not physical_bound_issues:
    print("PASS: All duration variables are within physical bounds.")
else:
    physical_bounds_report = pd.concat(
        physical_bound_issues,
        ignore_index=True
    )

    print(
        f"FAIL: {len(physical_bounds_report)} "
        "physical-bound issue(s) found."
    )

    display(physical_bounds_report)

===== Physical Bounds Check =====
FAIL: 35 physical-bound issue(s) found.


,Participant Id,Date,Home Duration,Check,Issue
0,47fjgl9q,2023-05-24,24.012867,Physical Bounds,Home Duration must be between 0 and 24 hours
1,4nki8mrn,2023-09-30,24.262500,Physical Bounds,Home Duration must be between 0 and 24 hours
2,4nki8mrn,2023-10-02,24.020647,Physical Bounds,Home Duration must be between 0 and 24 hours
3,6z4elhxv,2023-04-13,24.002778,Physical Bounds,Home Duration must be between 0 and 24 hours
4,6z4elhxv,2023-04-15,24.150000,Physical Bounds,Home Duration must be between 0 and 24 hours
5,8nwx7s1n,2023-06-19,24.055556,Physical Bounds,Home Duration must be between 0 and 24 hours
6,9kfpqf5f,2023-08-15,24.011111,Physical Bounds,Home Duration must be between 0 and 24 hours
7,cqbxn7in,2023-05-13,24.027778,Physical Bounds,Home Duration must be between 0 and 24 hours
8,cqbxn7in,2023-05-14,24.025198,Physical Bounds,Home Duration must be between 0 and 24 hours
9,cqbxn7in,2023-05-26,24.038889,Physical Bounds,Home Duration must be between 0 and 24 hours


Observation Duration: this chunk checks that observation day + observation night = observation duration (approximately). 

In [7]:
required_cols = ["Participant Id", "Date", "Obs Day", "Obs Night", "Obs Duration"]
missing_cols = [col for col in required_cols if col not in df.columns]

print("===== Observation Duration Consistency Check =====")

if missing_cols:
    print("FAIL: Missing required column(s):")
    print(missing_cols)
else:
    tolerance = 0.01

    invalid_obs_duration = df[
        df["Obs Day"].notna()
        & df["Obs Night"].notna()
        & df["Obs Duration"].notna()
        & ((df["Obs Day"] + df["Obs Night"] - df["Obs Duration"]).abs() > tolerance)
    ]

    if invalid_obs_duration.empty:
        print("PASS: Obs Day + Obs Night is approximately equal to Obs Duration.")
    else:
        print(f"FAIL: {len(invalid_obs_duration)} row(s) where Obs Day + Obs Night does not equal Obs Duration.")
        display(invalid_obs_duration[
            ["Participant Id", "Date", "Obs Day", "Obs Night", "Obs Duration"]
        ])

===== Observation Duration Consistency Check =====
PASS: Obs Day + Obs Night is approximately equal to Obs Duration.


Call Counts vs. Durations Check: this chunk will flag if the incoming call count = 0 and incoming call duration is > 0 because you cannot accumulate call duration with no calls.

In [8]:
required_cols = [
    "Participant Id", "Date",
    "Incoming Call Count", "Incoming Call Duration",
    "Outgoing Call Count", "Outgoing Call Duration"
]
missing_cols = [col for col in required_cols if col not in df.columns]

print("===== Call Counts vs. Durations Check =====")

if missing_cols:
    print("FAIL: Missing required column(s):")
    print(missing_cols)
else:
    invalid_incoming_calls = df[
        (df["Incoming Call Count"] == 0)
        & (df["Incoming Call Duration"] > 0)
    ]

    invalid_outgoing_calls = df[
        (df["Outgoing Call Count"] == 0)
        & (df["Outgoing Call Duration"] > 0)
    ]

    if invalid_incoming_calls.empty and invalid_outgoing_calls.empty:
        print("PASS: No call count/duration inconsistencies found.")
    else:
        print("FAIL: Call count/duration inconsistencies found.")

        if not invalid_incoming_calls.empty:
            print(f"\nIncoming calls: {len(invalid_incoming_calls)} row(s) where count = 0 but duration > 0")
            display(invalid_incoming_calls[
                ["Participant Id", "Date", "Incoming Call Count", "Incoming Call Duration"]
            ])

        if not invalid_outgoing_calls.empty:
            print(f"\nOutgoing calls: {len(invalid_outgoing_calls)} row(s) where count = 0 but duration > 0")
            display(invalid_outgoing_calls[
                ["Participant Id", "Date", "Outgoing Call Count", "Outgoing Call Duration"]
            ])

===== Call Counts vs. Durations Check =====
PASS: No call count/duration inconsistencies found.


Text Counts vs. Lengths Check: this will flag if the outgoing text count = 0 and outgoing text length is > 0.

In [9]:
required_cols = [
    "Participant Id", "Date",
    "Outgoing Text Count", "Outgoing Text Length"
]
missing_cols = [col for col in required_cols if col not in df.columns]

print("===== Text Counts vs. Lengths Check =====")

if missing_cols:
    print("FAIL: Missing required column(s):")
    print(missing_cols)
else:
    invalid_outgoing_texts = df[
        (df["Outgoing Text Count"] == 0)
        & (df["Outgoing Text Length"] > 0)
    ]

    if invalid_outgoing_texts.empty:
        print("PASS: No outgoing text count/length inconsistencies found.")
    else:
        print(f"FAIL: {len(invalid_outgoing_texts)} row(s) where outgoing text count = 0 but outgoing text length > 0.")
        display(invalid_outgoing_texts[
            ["Participant Id", "Date", "Outgoing Text Count", "Outgoing Text Length"]
        ])

===== Text Counts vs. Lengths Check =====
PASS: No outgoing text count/length inconsistencies found.


GPS Movement Consistency: this will flag if distance traveled = 0 and flight distance average is > 0.

In [11]:

required_cols = [
    "Participant Id", "Date",
    "Distance Traveled", "Flight Distance Average"
]
missing_cols = [col for col in required_cols if col not in df.columns]

print("===== GPS Movement Consistency Check =====")

if missing_cols:
    print("FAIL: Missing required column(s):")
    print(missing_cols)
else:
    invalid_gps_movement = df[
        (df["Distance Traveled"] == 0)
        & (df["Flight Distance Average"] > 0)
    ]

    if invalid_gps_movement.empty:
        print("PASS: No GPS movement inconsistencies found.")
    else:
        print(f"FAIL: {len(invalid_gps_movement)} row(s) where Distance Traveled = 0 but Flight Distance Average > 0.")
        display(invalid_gps_movement[
            [
                "Participant Id",
                "Date",
                "Distance Traveled",
                "Flight Distance Average"
            ]
        ])

===== GPS Movement Consistency Check =====
PASS: No GPS movement inconsistencies found.
